# Phase 3 orchestrator -- end to end

Live check against the real `app/orchestrator/orchestrator.py` module (not
a redefinition of it) -- companion to `phase3_graph.ipynb`, which stays as
the original prototyping notebook, unchanged.

Requires the MCP server running separately first: `uv run python -c "from app.mcp_server.server import mcp; import uvicorn; uvicorn.run(mcp.http_app(), host='127.0.0.1', port=8001)"`

In [1]:
import sys
sys.path.insert(0, "..")

import os
import uuid
from datetime import datetime

from langchain_core.messages import HumanMessage

from app.config import GCP_PROJECT_ID, get_secret
import app.orchestrator.orchestrator as orchestrator

os.environ["ANTHROPIC_API_KEY"] = get_secret("anthropic-api-key", GCP_PROJECT_ID)

await orchestrator.init_orchestrator()
print("MCP tools loaded:", list(orchestrator.MCP_TOOLS))

MCP tools loaded: ['ping']


## Run it end to end

Real Claude calls, real telemetry write -- via `astream`, matching the
gateway's actual invocation pattern (`.claude/rules/gateway.md`).

In [ ]:
question = "In one sentence, what is the Instacart dataset used for?"
initial_state = {
    "question": question,
    "conversation_id": str(uuid.uuid4()),
    "user_id": "notebook-test-user",
    "turn_started_at": datetime.now(),
    "filter_context": [],
    "active_page": None,
    "image_base64": None,
    "history_messages": [],
    "messages": [HumanMessage(content=question)],
    "tool_calls": [],
    "iteration_count": 0,
    "verification_retry_count": 0,
    "length_retry_count": 0,
    "verified": False,
    "bytes_consumed": 0,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "llm_calls": 0,
    "errors": [],
    "cancelled": False,
    "needs_approval": False,
    "pending_queries": [],
    "deferred_dax": [],
    "estimated_cost": None,
    "cost_cap_exceeded": False,
    "iteration_cap_hit": False,
    "answer_submitted": False,
    "answer_markdown": "",
    "chart_url": None,
    "sources": [],
    "all_prose_numeric_claims": [],
    "suggested_follow_ups": [],
}

final_state = None
async for chunk in orchestrator.graph.astream(initial_state, stream_mode=["updates", "values"]):
    kind, data = chunk
    if kind == "values":
        final_state = data
        continue
    node = next(iter(data))
    print(f"-> node ran: {node}")

print()
print("answer_markdown:", final_state["answer_markdown"])
print("verified:", final_state["verified"])
print("all_prose_numeric_claims:", final_state["all_prose_numeric_claims"])
print("suggested_follow_ups:", final_state["suggested_follow_ups"])
print("llm_calls:", final_state["llm_calls"])
print("message count:", len(final_state["messages"]))
assert final_state["verified"] is True
assert len(final_state["answer_markdown"]) > 0
print()
print("PASS -- real app/orchestrator/orchestrator.py ran end to end, telemetry written")